In [2]:
import os
import sys
import re
import json
import zipfile
import shutil
import sqlite3
import pandas as pd
from collections import defaultdict

In [8]:
folder = os.path.expanduser('~/data/IoT/11 Nov 2020')
db_path = '/users/PAS1266/ralston168/projects/IoT-ML/notebooks/logs.db'

# Find all zip files
zip_files = sorted([f for f in os.listdir(folder) if f.endswith('.zip')])

if not zip_files:
    print(f'ERROR: No zip files found in {folder}')
    sys.exit(1)

print(f'Found {len(zip_files)} zip files in {folder}:')
for z in zip_files:
    size = os.path.getsize(os.path.join(folder, z))
    print(f'  {z} ({size / (1024*1024):.2f} MB)')

print(f'\nOutput database: {db_path}')

Found 38 zip files in /users/PAS1266/ralston168/data/IoT/11 Nov 2020:
  02-31.132.3c0b.bc11.zip (32.14 MB)
  03-31.132.d063.8013.zip (0.00 MB)
  04-46.246.fe1d.9416.zip (0.00 MB)
  05-76.73.fa3a.5d9f.zip (0.00 MB)
  06-92.240.6ddd.5c03.zip (178.17 MB)
  07-213.184.2d21.9dd9.zip (105.55 MB)
  08-154.16.513e.1a2a.zip (212.53 MB)
  09-213.184.50de.192b.zip (125.46 MB)
  10-173.225.d4bd.03e4.zip (189.90 MB)
  11-209.200.ae9c.e04e.zip (181.08 MB)
  12-213.184.ecbf.7293.zip (131.72 MB)
  13-93.190.abcd.1234.zip (3.21 MB)
  14-94.229.2f7c.94dd.zip (194.57 MB)
  16-77.78.541a.814d.zip (75.91 MB)
  17-77.78.e513.a78b.zip (0.00 MB)
  18-94.229.d716.62a3.zip (175.99 MB)
  19-108.62.acb9.b7ae.zip (37.80 MB)
  20-109.200.3d7c.3d77.zip (91.16 MB)
  21-109.200.2806.636a.zip (0.00 MB)
  22-154.16.4513.6c50.zip (0.00 MB)
  23-173.225.1539.32ec.zip (65.15 MB)
  24-173.225.c128.a067.zip (0.00 MB)
  25-173.225.f157.adf6.zip (0.00 MB)
  26-173.225.49be.6699.zip (0.00 MB)
  27-154.16.f0b8.45c9.zip (219.76 M

In [9]:
def get_log_type(filename):
    """
    Extract log type from filename.
    e.g. '5.175.8717.aad2_00001_20170724102704-conn.logreplaced.log' -> 'conn'
    e.g. '5.175.8717.aad2_00001_20170724102704-packet_filter.logreplaced.log' -> 'packet_filter'
    """
    match = re.search(r'-([a-zA-Z_]+)\.logreplaced\.log$', filename)
    if match:
        return match.group(1)
    return None


temp_folder = os.path.join(folder, '_temp_extracted')
table_written = set()  # track which tables have been created (to avoid replacing on append)

total_zips_processed = 0
total_rows_inserted = 0

conn = sqlite3.connect(db_path)

try:
    for i, zip_filename in enumerate(zip_files, 1):
        print(f'\n[{i}/{len(zip_files)}] Processing: {zip_filename}')

        try:
            # -- Unzip --
            os.makedirs(temp_folder, exist_ok=True)
            zip_path = os.path.join(folder, zip_filename)

            with zipfile.ZipFile(zip_path, 'r') as z:
                z.extractall(temp_folder)
                num_extracted = len(z.namelist())
                print(f'  Unzipped: {num_extracted} files')

            # -- Group files by log type (walk recursively — zips may contain subdirs) --
            log_groups = defaultdict(list)
            unrecognised = 0

            for dirpath, _, filenames in os.walk(temp_folder):
                for filename in filenames:
                    log_type = get_log_type(filename)
                    if log_type:
                        log_groups[log_type].append(os.path.join(dirpath, filename))
                    else:
                        unrecognised += 1

            if not log_groups:
                print(f'  WARNING: No recognisable log files found in {zip_filename}, skipping')
                continue

            if unrecognised:
                print(f'  Skipping {unrecognised} unrecognised files')

            # -- Parse and insert each log type into the database --
            for log_type, filepaths in sorted(log_groups.items()):
                filepaths.sort()  # sort chronologically by timestamp in filename
                table_name = f'all_{log_type}'
                all_logs = []

                for filepath in filepaths:
                    with open(filepath, 'r', errors='replace') as f:
                        for line in f:
                            line = line.strip()
                            if line:
                                try:
                                    all_logs.append(json.loads(line))
                                except json.JSONDecodeError:
                                    pass

                if not all_logs:
                    continue

                df = pd.DataFrame(all_logs)

                # Flatten list columns for SQLite compatibility
                for col in df.columns:
                    if df[col].apply(lambda x: isinstance(x, list)).any():
                        df[col] = df[col].apply(lambda x: ', '.join(map(str, x)) if isinstance(x, list) else x)

                if_exists = 'replace' if table_name not in table_written else 'append'
                df.to_sql(table_name, conn, if_exists=if_exists, index=False)
                table_written.add(table_name)

                print(f'  Inserted {len(df):6} rows → {table_name}')
                total_rows_inserted += len(df)

            total_zips_processed += 1

        except zipfile.BadZipFile:
            print(f'  SKIPPED (bad zip): {zip_filename}')

        except Exception as e:
            print(f'  ERROR processing {zip_filename}: {e}')

        finally:
            # Always clean up temp folder even if something goes wrong
            if os.path.exists(temp_folder):
                shutil.rmtree(temp_folder)
                print(f'  Cleaned up temp files')

finally:
    conn.close()

print(f'\n' + '=' * 60)
print(f'Zip files processed: {total_zips_processed}/{len(zip_files)}')
print(f'Total rows inserted: {total_rows_inserted}')
print(f'Tables created: {sorted(table_written)}')



[1/38] Processing: 02-31.132.3c0b.bc11.zip
  Unzipped: 10088 files
  Skipping 6 unrecognised files
  Inserted 669659 rows → all_conn
  Inserted    212 rows → all_dns
  Inserted    133 rows → all_files
  Inserted      7 rows → all_http
  Inserted     13 rows → all_kerberos
  Inserted     90 rows → all_ntp
  Inserted   2319 rows → all_packet_filter
  Inserted   2173 rows → all_reporter
  Inserted  10309 rows → all_sip
  Inserted    345 rows → all_snmp
  Inserted     33 rows → all_ssl
  Inserted      2 rows → all_tunnel
  Inserted    538 rows → all_weird
  Cleaned up temp files

[2/38] Processing: 03-31.132.d063.8013.zip
  SKIPPED (bad zip): 03-31.132.d063.8013.zip
  Cleaned up temp files

[3/38] Processing: 04-46.246.fe1d.9416.zip
  SKIPPED (bad zip): 04-46.246.fe1d.9416.zip
  Cleaned up temp files

[4/38] Processing: 05-76.73.fa3a.5d9f.zip
  SKIPPED (bad zip): 05-76.73.fa3a.5d9f.zip
  Cleaned up temp files

[5/38] Processing: 06-92.240.6ddd.5c03.zip
  Unzipped: 54918 files
  Skipping 1

In [6]:
# Add time_bucket_id column to each table (daily buckets from min timestamp)
# Works independently on an existing db — discovers tables from the db itself
conn = sqlite3.connect("logs.db")

cursor = conn.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")
tables = [row[0] for row in cursor.fetchall()]

for table_name in tables:
    print(f'Processing {table_name}...')
    df = pd.read_sql(f'SELECT * FROM "{table_name}"', conn)

    if 'time_bucket_id' in df.columns:
        print(f'  {table_name} already has time_bucket_id; skipped')
        continue

    if 'ts' in df.columns:
        df = df.sort_values('ts')
        min_ts = df['ts'].min()
        df['time_bucket_id'] = ((df['ts'] - min_ts) // 86400).astype(int)
        df.to_sql(table_name, conn, if_exists='replace', index=False)
        print(f'  Updated {table_name} with {len(df)} rows and time_bucket_id')
    else:
        print(f'  {table_name} has no ts column; skipped')

conn.close()
print('\nDone!')

Processing all_conn...
  Updated all_conn with 41863898 rows and time_bucket_id
Processing all_dns...
  Updated all_dns with 374757 rows and time_bucket_id
Processing all_dpd...
  Updated all_dpd with 39806 rows and time_bucket_id
Processing all_files...
  Updated all_files with 887260 rows and time_bucket_id
Processing all_http...
  Updated all_http with 7 rows and time_bucket_id
Processing all_kerberos...
  Updated all_kerberos with 13 rows and time_bucket_id
Processing all_ntp...
  Updated all_ntp with 90 rows and time_bucket_id
Processing all_packet_filter...
  Updated all_packet_filter with 8223 rows and time_bucket_id
Processing all_reporter...
  Updated all_reporter with 2173 rows and time_bucket_id
Processing all_sip...
  Updated all_sip with 10309 rows and time_bucket_id
Processing all_snmp...
  Updated all_snmp with 345 rows and time_bucket_id
Processing all_ssl...
  Updated all_ssl with 33 rows and time_bucket_id
Processing all_tunnel...
  Updated all_tunnel with 2 rows and 